# Bias, variance and regularisation

Notebook 11 showed that a flexible model fits the training data beautifully and then fails on new data. This notebook explains why, in two numbers, and then fixes it with a penalty on the weights. Ridge, Lasso and ElasticNet are the three standard penalties.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import (
    ElasticNet,
    ElasticNetCV,
    Lasso,
    LassoCV,
    LinearRegression,
    Ridge,
    RidgeCV,
    lasso_path,
)
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from common.datasets import (
    cap_housing_outliers,
    load_california_housing,
    make_noisy_curve,
    make_sparse_regression,
    true_curve,
)

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Imagine you could collect many different training sets from the same source and fit the same kind of model to each. The fitted curves would differ from set to set. Two things then decide how wrong you are on average.

- **Bias:** the average of all those curves is still off target. The model type cannot bend the right way. A straight line through a wave has high bias.
- **Variance:** the curves scatter widely around their own average, because each fit chases the quirks of its own sample. A degree-15 polynomial has high variance.

On top of both sits irreducible noise, which no model removes.

Simple models have high bias and low variance. Flexible models have low bias and high variance. Regularisation lets you keep a flexible model and buy back variance with a small, controlled amount of bias.

## 2. Maths

Predict at a point $x_0$ with $y=f(x_0)+\varepsilon$, $\operatorname{Var}(\varepsilon)=\sigma^2$, and a model $\hat f$ fitted to a random training set. The expected squared error splits into three parts:

$$\mathbb E\bigl[(y-\hat f(x_0))^2\bigr]=\underbrace{\bigl(\mathbb E[\hat f(x_0)]-f(x_0)\bigr)^2}_{\text{bias}^2}+\underbrace{\operatorname{Var}\bigl(\hat f(x_0)\bigr)}_{\text{variance}}+\underbrace{\sigma^2}_{\text{noise}}$$

The expectation is over training sets, which we can only imagine in real life. In a simulation we can draw hundreds of them, so we will.

**Penalised cost.** Add a penalty on the weights (never on the intercept) to the MSE:

| Method | Cost | Effect |
|---|---|---|
| Ridge (L2) | $\text{MSE}+\alpha\sum_j w_j^2$ | shrinks all weights smoothly |
| Lasso (L1) | $\text{MSE}+\alpha\sum_j \lvert w_j\rvert$ | shrinks and sets some exactly to zero |
| ElasticNet | $\text{MSE}+\alpha\bigl[\rho\sum\lvert w_j\rvert+\tfrac{1-\rho}{2}\sum w_j^2\bigr]$ | a blend, $\rho$ is `l1_ratio` |

Ridge has a closed form, $\hat{\mathbf w}=(X^\top X+\alpha I)^{-1}X^\top\mathbf y$. Adding $\alpha I$ also makes the matrix invertible when columns are collinear. Lasso has no closed form, and is solved by coordinate descent.

**Where the penalties come from.** In `02_probability/03` a Gaussian prior on the weights gave an L2 penalty through MAP estimation. A Laplace prior (sharply peaked at zero, with heavy tails) gives an L1 penalty. So $\alpha$ is how strongly you believe the weights are small: for Gaussian noise $\sigma^2$ and a prior with variance $\tau^2$ the match is $\alpha=\sigma^2/\tau^2$.

**Why L1 gives zeros.** Suppose the features are uncorrelated and scaled so the unpenalised weight is $\hat w$. With threshold $\lambda$ (the penalty strength in the right units) the penalised answers have simple forms:

$$\text{Ridge: } \frac{\hat w}{1+\lambda}\qquad\text{Lasso: } \operatorname{sign}(\hat w)\max(\lvert\hat w\rvert-\lambda,\,0)$$

Ridge divides every weight by the same factor, so none reach zero. Lasso subtracts a fixed amount and clips at zero, so any weight smaller than $\lambda$ is removed. Geometrically, the L1 constraint region is a diamond with corners on the axes, and the cost contours usually touch it at a corner, where some weights are zero. The L2 region is a ball with no corners.

## 3. Bias and variance by simulation

Draw 200 training sets of 30 points from the noisy curve (noise sd 0.3, so $\sigma^2=0.09$). Fit each, predict on a fixed grid, and measure the three terms against the true curve.

In [ ]:
grid = np.linspace(-2.8, 2.8, 100)
G = pd.DataFrame({"x": grid})
f_true = true_curve(grid)
NOISE = 0.3


def simulate(model_factory, n_sets: int = 200, n: int = 30) -> tuple[float, float]:
    preds = np.empty((n_sets, len(grid)))
    for s in range(n_sets):
        Xs, ys = make_noisy_curve(n_samples=n, noise=NOISE, seed=1000 + s)
        preds[s] = model_factory().fit(Xs, ys).predict(G)
    bias2 = np.mean((preds.mean(axis=0) - f_true) ** 2)
    var = np.mean(preds.var(axis=0))
    return float(bias2), float(var)


def poly(degree: int, est=None):
    return make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        StandardScaler(),
        est if est is not None else LinearRegression(),
    )


rows = []
for d in (1, 2, 3, 5, 8, 12):
    b2, v = simulate(lambda d=d: poly(d))
    rows.append(
        {"degree": d, "bias2": b2, "variance": v, "noise": NOISE**2, "total": b2 + v + NOISE**2}
    )
table = pd.DataFrame(rows).set_index("degree")
table

In [ ]:
table[["bias2", "variance"]].plot(logy=True, marker="o")
plt.ylabel("squared error (log scale)")
plt.show()

Read the table down the degrees. Bias squared falls from about 0.45 (a line through a wave) to about 0.001 at degree 5. Variance starts near 0.04 and explodes: 1.2 at degree 8 and over 100 at degree 12, which is a thousand times the noise. Total error is lowest at degree 5 (0.13, against a floor of 0.09). Degree 12 averages worse than a straight line, because a few of its 200 fits swing far off the curve.

Look at it directly: 25 fitted curves for a stiff and a flexible model, with the true curve in black.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, d in zip(axes, (1, 3, 12)):
    for s in range(25):
        Xs, ys = make_noisy_curve(n_samples=30, noise=NOISE, seed=1000 + s)
        ax.plot(grid, poly(d).fit(Xs, ys).predict(G), c="tab:blue", alpha=0.25, lw=1)
    ax.plot(grid, f_true, c="k")
    ax.set_ylim(-3, 3)
    ax.set_title(f"degree {d}")
plt.show()

Degree 1: the bundle is tight but misses the wave (bias). Degree 12: the bundle is centred on the truth but wildly spread (variance). Degree 3 is both tight and close.

Now keep degree 12 and add a Ridge penalty. Strength $\alpha$ trades one for the other.

In [ ]:
rows = []
for a in (0.001, 0.1, 1, 10, 100):
    b2, v = simulate(lambda a=a: poly(12, Ridge(alpha=a)))
    rows.append({"alpha": a, "bias2": b2, "variance": v, "total": b2 + v + NOISE**2})
pd.DataFrame(rows).set_index("alpha")

Unpenalised, degree 12 has a total error above 100. A penalty of 0.1 brings it to 0.16 by cutting variance from over 100 to 0.05, at the price of a little bias (0.02). Push $\alpha$ to 100 and bias takes over as the curve flattens towards a line. The best $\alpha$ is in between, and cross-validation will find it (section 7).

## 4. From scratch

Ridge is one line of linear algebra. Lasso needs coordinate descent: update one weight at a time, holding the rest fixed, using the soft-threshold from the maths section. Both below work on centred data, so the intercept is left out and never penalised.

First, the data for the rest of the notebook: 50 features of which only 5 matter, 80 rows to train on and 400 to test. This is the case where plain least squares struggles.

In [ ]:
Xa, ya, w_true = make_sparse_regression(
    n_samples=480, n_features=50, n_informative=5, noise=3.0, seed=SEED
)
Xtr, Xte, ytr, yte = train_test_split(Xa, ya, train_size=80, random_state=SEED)
print("true non-zero weights:", w_true[:5], "...", "rest are exactly 0")


def centre(Xm: np.ndarray, ym: np.ndarray):
    return Xm - Xm.mean(axis=0), ym - ym.mean()


def ridge_scratch(Xm: np.ndarray, ym: np.ndarray, alpha: float) -> np.ndarray:
    Xc, yc = centre(Xm, ym)
    return np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ yc)


def soft_threshold(rho: float, lam: float) -> float:
    return np.sign(rho) * max(abs(rho) - lam, 0.0)


def lasso_scratch(Xm: np.ndarray, ym: np.ndarray, alpha: float, sweeps: int = 200) -> np.ndarray:
    # minimises (1 / 2n) * ||y - Xw||^2 + alpha * ||w||_1, the same objective as sklearn
    Xc, yc = centre(Xm, ym)
    n, d = Xc.shape
    w = np.zeros(d)
    col_sq = (Xc**2).sum(axis=0) / n
    for _ in range(sweeps):
        for j in range(d):
            resid_j = yc - Xc @ w + Xc[:, j] * w[j]  # residual with feature j removed
            w[j] = soft_threshold(Xc[:, j] @ resid_j / n, alpha) / col_sq[j]
    return w


Xn, yn = Xtr.to_numpy(), ytr.to_numpy()
print(
    "ridge matches sklearn:",
    np.allclose(ridge_scratch(Xn, yn, 10.0), Ridge(alpha=10.0).fit(Xn, yn).coef_),
)
w_l = lasso_scratch(Xn, yn, 0.3)
print("lasso matches sklearn:", np.allclose(w_l, Lasso(alpha=0.3).fit(Xn, yn).coef_, atol=1e-4))
print("lasso non-zeros:", np.count_nonzero(w_l))

## 5. The library version on 5 real features out of 50

Fit OLS, Ridge, Lasso and ElasticNet. I judge them on three things: held-out error, how far the weights are from the truth, and how many weights are non-zero. The data are already on one scale, but in real work, standardise first (section 8 shows why).

In [ ]:
models = {
    "OLS": LinearRegression(),
    "Ridge(10)": Ridge(alpha=10),
    "Lasso(0.3)": Lasso(alpha=0.3),
    "ElasticNet(0.3, 0.5)": ElasticNet(alpha=0.3, l1_ratio=0.5),
}
rows, coefs = [], {}
for name, m in models.items():
    m.fit(Xtr, ytr)
    coefs[name] = m.coef_
    rows.append(
        {
            "model": name,
            "test_MSE": mean_squared_error(yte, m.predict(Xte)),
            "weight_error": np.linalg.norm(m.coef_ - w_true),
            "non_zero": int(np.count_nonzero(np.abs(m.coef_) > 1e-8)),
        }
    )
pd.DataFrame(rows).set_index("model")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 2.8), sharey=True)
for ax, (name, c) in zip(axes, coefs.items()):
    ax.stem(c, markerfmt=".", basefmt=" ")
    ax.scatter(range(5), w_true[:5], c="r", s=14, zorder=3)
    ax.set_title(name)
plt.show()

The red dots are the true weights. OLS has 50 non-zero weights, and 45 of them are noise. Ridge shrinks everything, the noise weights become small but none are zero. Lasso zeros most of the 45 irrelevant weights and keeps the real ones. For 5 real features in 50, Lasso (or ElasticNet) is the right first choice. If instead all 50 features carry a little signal, Ridge usually wins.

## 6. L1 against L2 on one weight

The soft-threshold picture, drawn for the uncorrelated case from the maths section.

In [ ]:
w_hat = np.linspace(-3, 3, 200)
lam = 1.0
plt.plot(w_hat, w_hat, c="grey", ls=":", label="unpenalised")
plt.plot(w_hat, w_hat / (1 + lam), label="Ridge: scale down")
plt.plot(
    w_hat, np.sign(w_hat) * np.maximum(np.abs(w_hat) - lam, 0), label="Lasso: subtract and clip"
)
plt.xlabel("unpenalised weight")
plt.ylabel("penalised weight")
plt.legend()
plt.show()

Ridge is a straight line with a smaller slope. Lasso has a flat stretch at zero: every weight with $|\hat w|<\lambda$ lands exactly there. That flat stretch is the sparsity.

## 7. Tuning $\alpha$ with cross-validation

$\alpha$ is a hyperparameter, so it is chosen on training data only, by cross-validation. The `CV` classes do the search with a built-in path, which is faster than a manual grid. Standardising goes inside the pipeline.

In [ ]:
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 40))).fit(Xtr, ytr)
lasso_cv = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=SEED, alphas=60)).fit(
    Xtr, ytr
)
enet_cv = make_pipeline(
    StandardScaler(),
    ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8, 1.0], cv=5, random_state=SEED, alphas=60),
).fit(Xtr, ytr)

for name, m in [("RidgeCV", ridge_cv), ("LassoCV", lasso_cv), ("ElasticNetCV", enet_cv)]:
    est = m[-1]
    print(f"{name:>12}: alpha={est.alpha_:.3f}", end="")
    if hasattr(est, "l1_ratio_"):
        print(f", l1_ratio={est.l1_ratio_}", end="")
    print(
        f", non-zero={np.count_nonzero(est.coef_)}, test MSE={mean_squared_error(yte, m.predict(Xte)):.2f}"
    )
print("OLS test MSE:", mean_squared_error(yte, LinearRegression().fit(Xtr, ytr).predict(Xte)))
print("noise floor  :", 3.0**2)

Lasso's cross-validated error curve shows the same bowl as every other tuning curve: too little penalty overfits, too much throws away real signal.

In [ ]:
est = lasso_cv[-1]
plt.plot(est.alphas_, est.mse_path_.mean(axis=1))
plt.axvline(est.alpha_, ls="--", c="grey")
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("CV MSE")
plt.show()

### Regularisation path

Plot every weight as $\alpha$ grows from zero. This is the clearest picture of what each penalty does. The 5 real features are in colour and the 45 noise features in grey.

In [ ]:
Xs = StandardScaler().fit_transform(Xtr)
alphas_l, coefs_l, _ = lasso_path(Xs, ytr.to_numpy() - ytr.mean(), alphas=60)
alphas_r = np.logspace(-2, 3, 60)
coefs_r = np.array([Ridge(alpha=a).fit(Xs, ytr).coef_ for a in alphas_r]).T

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, al, co, title in [
    (a1, alphas_r, coefs_r, "Ridge path"),
    (a2, alphas_l, coefs_l, "Lasso path"),
]:
    for j in range(50):
        ax.plot(al, co[j], c="tab:red" if j < 5 else "lightgrey", lw=1.5 if j < 5 else 0.8)
    ax.set_xscale("log")
    ax.set_xlabel("alpha")
    ax.set_title(title)
a1.set_ylabel("weight")
plt.show()

In the Ridge path, all weights glide towards zero together and never touch it. In the Lasso path, the grey noise weights drop to zero early and stay there, while the real ones survive until the penalty is large. That early separation is why Lasso works as feature selection.

## 8. Real data: California housing with 164 features

Back to real data. Degree-3 polynomial features on 8 inputs give 164 columns, trained on only 400 rows. This is plain overfitting territory.

In [ ]:
Xh, yh = load_california_housing(n_samples=1500, seed=SEED)
Xh = cap_housing_outliers(Xh)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, train_size=400, random_state=SEED)


def housing(est):
    return make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(), est)


fits = {
    "OLS": housing(LinearRegression()),
    "RidgeCV": housing(RidgeCV(alphas=np.logspace(-1, 4, 30))),
    "LassoCV": housing(LassoCV(cv=5, random_state=SEED, alphas=30, max_iter=20000)),
}
for name, m in fits.items():
    m.fit(Xh_tr, yh_tr)
    coef = m[-1].coef_
    print(
        f"{name:>8}: test R^2 = {m.score(Xh_te, yh_te):8.3f}   non-zero weights = {np.count_nonzero(coef)}"
    )

OLS is badly wrong on new data: the weights have fitted noise. Ridge fixes most of it. Lasso does about as well with only a fraction of the 164 columns, which makes it easier to explain.

## 9. What breaks

**Scale.** The penalty treats all weights equally, so it assumes all features are on a comparable scale. Shrink one real feature by a factor of 100 and its weight must become 100 times larger to do the same job, which the penalty then punishes.

In [ ]:
Xtr_small = Xtr.copy()
Xtr_small["f0"] = Xtr["f0"] * 0.01
raw = Ridge(alpha=10).fit(Xtr_small, ytr).coef_[0] * 0.01
scaled = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xtr_small, ytr)
print("true weight of f0              :", w_true[0])
print("Ridge, f0 shrunk, no scaling   :", raw, "(in original units)")
print("Ridge, f0 shrunk, with scaling :", scaled[-1].coef_[0] / Xtr["f0"].std())

Without scaling, the model almost deletes a feature that matters, purely because of its units. With scaling, it keeps it. Always standardise before a penalised model.

**Correlated features.** Lasso tends to keep one of a group of near-duplicates and drop the others, and which one it keeps is unstable. ElasticNet spreads the weight across the group.

In [ ]:
z = rng.normal(size=300)
Xc = np.column_stack([z + rng.normal(0, 0.05, 300) for _ in range(3)] + [rng.normal(size=300)])
yc = 3 * z + rng.normal(0, 1, 300)
print("Lasso      :", Lasso(alpha=0.1).fit(Xc, yc).coef_)
print("ElasticNet :", ElasticNet(alpha=0.1, l1_ratio=0.3).fit(Xc, yc).coef_)

Note that the earlier CV search chose `l1_ratio=1.0`, so ElasticNet collapsed to Lasso. That is the right call when features are independent, as they are here.

The three near-copies of $z$ should share a total weight of about 3. Lasso piles it on one or two of them. ElasticNet gives each about 1.

**Other limits.** Penalties do not fix a wrong model: if the shape is wrong, bias stays. They need $\alpha$ tuned on held-out data, and Lasso's selection is only as stable as your sample is large. The intercept is never penalised.

A rough rule I use:

- All features probably matter a bit, or they are correlated: Ridge.
- Few features probably matter, and you want a short list: Lasso.
- Correlated groups and you still want sparsity: ElasticNet.

## Summary

- Expected error is bias squared plus variance plus noise. Flexibility trades the first for the second.
- Regularisation adds a penalty on weights. It increases bias a little and cuts variance a lot, so the total falls.
- L2 shrinks, L1 shrinks and zeros, ElasticNet blends. L2 is a Gaussian prior and L1 is a Laplace prior.
- Standardise first. Tune $\alpha$ by cross-validation inside a pipeline.

Next: `regression_intuition_test.ipynb`.